In [1]:
import torch
from torch import nn
from dataset import Rice2
from torch.utils.data import DataLoader, random_split
from torch.optim import Adam
from torch.utils.data import Subset

from utility import plot_samples, denorm, plot_losses, to_img, plot_test_samples
from losses import vae_loss, weighted_l1_loss
from models import Autoencoder

import time
import os

from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
model = Autoencoder(latent_dim=512).to(device)
optimizer = Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",        # wir wollen den Loss minimieren
    factor=0.5,        # lr wird halbiert wenn Plateau
    patience=2,        # ... nach 5 Epochen ohne Verbesserung
    threshold=0.1
)

num_epochs = 50
batch_size = 50
beta = 0.01
train_losses, val_losses = [], []
train_recon, val_recon = [], []
train_kl, val_kl = [], []
val_psnrs, val_ssims = [], []
use_masks = False  # Masken als zusätzlichen Kanal verwenden
w_factor = 0.3            # Gewicht des maskierten L1-Terms
w_extra = 4.0             # Wolken-Extragewicht im maskierten L1
best_score = -1.0

# data_dir = "/home/christopher/sync/projekte/GNN_projekt/RICE_DATASET/RICE2"
data_dir = "/data/home/lec42675/cloud_removal/RICE_DATASET/RICE2"

# 1) Volles Dataset nur zur Bestimmung der Split-Indizes.
base_dataset = Rice2(data_dir, augment=False)
n_total = len(base_dataset)
 
n_train = int(0.64 * n_total)
n_val = int(0.16 * n_total)
n_test = n_total - n_train - n_val   # Rest, damit die Summe exakt aufgeht
 
# 2) Reproduzierbarer Split (fester Seed -> immer dieselbe Aufteilung).
split_generator = torch.Generator().manual_seed(42)
train_subset, val_subset, test_subset = random_split(
    base_dataset, [n_train, n_val, n_test], generator=split_generator
)
train_idx = train_subset.indices
val_idx = val_subset.indices
test_idx = test_subset.indices
 
# 3) Getrennte Dataset-Instanzen: Train augmentiert, Val/Test nicht.
train_data = Subset(Rice2(data_dir, augment=True), train_idx)
val_data = Subset(Rice2(data_dir, augment=False), val_idx)
test_data = Subset(Rice2(data_dir, augment=False), test_idx)
 
print(
    f"Gesamt: {n_total} | "
    f"Train: {len(train_data)} ({len(train_data)/n_total:.0%}) | "
    f"Val: {len(val_data)} ({len(val_data)/n_total:.0%}) | "
    f"Test: {len(test_data)} ({len(test_data)/n_total:.0%})"
)
 
# 4) DataLoader.
train_dataloader = DataLoader(
    train_data, batch_size=batch_size, shuffle=True,
    num_workers=4, pin_memory=True, persistent_workers=True,
)
val_dataloader = DataLoader(
    val_data, batch_size=batch_size, shuffle=False,
    num_workers=4, pin_memory=True, persistent_workers=True,
)
test_dataloader = DataLoader(
    test_data, batch_size=batch_size, shuffle=False,
    num_workers=4, pin_memory=True, persistent_workers=True,
)
 

Gesamt: 736 | Train: 471 (64%) | Val: 117 (16%) | Test: 148 (20%)


In [3]:
plot_dataset = Rice2(data_dir, augment=False)

# gewünschte Dateinamen (passe die Endung an deine echten Dateien an!)
wanted_names = ["548.png", "449.png", "282.png", "111.png", "22.png"]

cloudy_list, label_list, mask_list = [], [], []
for name in wanted_names:
    pos = plot_dataset.ids.index(name)   # Position dieses Dateinamens finden
    c, l, m = plot_dataset[pos]
    cloudy_list.append(c)
    label_list.append(l)
    mask_list.append(m)

fixed_batch = (torch.stack(cloudy_list), torch.stack(label_list), torch.stack(mask_list))

/data/home/lec42675/.venv/gpu-env/lib/python3.10/site-packages/torchvision/transforms/v2/functional/_deprecated.py:12: UserWarning: The function `to_tensor(...)` is deprecated and will be removed in a future release. Instead, please use `to_image(...)` followed by `to_dtype(..., dtype=torch.float32, scale=True)`.
  warnings.warn(


In [4]:
os.makedirs("checkpoints", exist_ok=True)

save_last_n_samples = 3   # Sample-Bilder der letzten N Epochen speichern
plot_every = 10
start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    total_loss = total_recon = total_kl = 0

    for cloudy, label, mask in train_dataloader:
        cloudy, label, mask = cloudy.to(device), label.to(device), mask.to(device)

        optimizer.zero_grad()
        if use_masks:
            # Masken als zusätzlichen Kanal an die Eingabe anhängen
            cloudy_with_mask = torch.cat([cloudy, mask], dim=1)
            recon, mu, logvar = model(cloudy_with_mask, label)
        else:
            recon, mu, logvar = model(cloudy, label)
        loss, recon_loss, kl_loss = vae_loss(recon, label, mu, logvar, beta)
        if use_masks:
            weighted_loss = weighted_l1_loss(recon, label, mask, extra=w_extra)
            loss = loss + w_factor * weighted_loss

        loss.backward()
        optimizer.step()
        total_loss  += loss.item()
        total_recon += recon_loss.item()
        total_kl    += kl_loss.item()
    train_losses.append(total_loss / len(train_dataloader))
    train_recon.append(total_recon / len(train_dataloader))
    train_kl.append(total_kl / len(train_dataloader))

    if scheduler is not None:
        scheduler.step(train_losses[-1])  # Scheduler basierend auf dem Trainingsloss aktualisieren

    model.eval()
    total_loss = total_recon = total_kl = 0
    psnr_total = 0.0
    ssim_total = 0.0
    n_images = 0

    with torch.no_grad():
        for cloudy, label, mask in val_dataloader:
            cloudy, label, mask = cloudy.to(device), label.to(device), mask.to(device)
            if use_masks:
                cloudy_with_mask = torch.cat([cloudy, mask], dim=1)
                recon, mu, logvar = model(cloudy_with_mask, label)
            else:
                recon, mu, logvar = model(cloudy, label)
            loss, recon_loss, kl_loss = vae_loss(recon, label, mu, logvar, beta)
            if use_masks:
                weighted_loss = weighted_l1_loss(recon, label, mask, extra=w_extra)
                loss = loss + w_factor * weighted_loss
            total_loss  += loss.item()
            total_recon += recon_loss.item()
            total_kl    += kl_loss.item()
            # PSNR / SSIM pro Bild
            for i in range(recon.size(0)):
                out_np = to_img(recon[i])
                gt_np = to_img(label[i])
                psnr_total += psnr_fn(gt_np, out_np, data_range=1.0)
                ssim_total += ssim_fn(gt_np, out_np, data_range=1.0, channel_axis=2)
                n_images += 1
    val_losses.append(total_loss / len(val_dataloader))
    val_recon.append(total_recon / len(val_dataloader))
    val_kl.append(total_kl / len(val_dataloader))
    val_psnrs.append(psnr_total / n_images)
    val_ssims.append(ssim_total / n_images)
    score = val_ssims[-1]

    if  score > best_score:
        torch.save(
            {
                "epoch": epoch,
                "generator": model.state_dict()
            },
            "checkpoints/best.pt",
        )
        best_score = score

    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch [{epoch+1}/{num_epochs}], Train: {train_losses[-1]:.4f}, "
          f"Val: {val_losses[-1]:.4f}, LR: {current_lr:.2e}"
          f"PSNR: {val_psnrs[-1]:.2f} | SSIM: {val_ssims[-1]:.4f}")
    
 #Sample-Outputs: regelmäßig zeigen, letzte paar Epochen speichern ----
    save_path = None
    if epoch >= num_epochs - save_last_n_samples:
        save_path = f"checkpoints/samples_epoch{epoch + 1}.png"
    if epoch % plot_every == 0 or epoch >= num_epochs - save_last_n_samples:
        plot_test_samples(model, fixed_batch, device, epoch, save_path=save_path)

    
 # ----------------------------------------------------------------------
# Nach dem Training: Plots speichern
# ----------------------------------------------------------------------
plot_losses(
    {
        "Total Loss": (train_losses, val_losses),
        "Reconstruction Loss": (train_recon, val_recon),
        "KL Loss": (train_kl, val_kl),
        "PSNR (dB)": (None, val_psnrs),
        "SSIM": (None, val_ssims),
    },
    model_name="CVAE",
    ncols=2,
    save_path="checkpoints/metrics.png",
)

duration = time.time() - start_time
print(
    f"The training took {duration:.2f} seconds "
    f"with {duration / num_epochs:.2f} seconds per epoch"
)
    


Epoch [1/50], Train: 0.3713, Val: 0.3982, LR: 1.00e-04PSNR: 14.44 | SSIM: 0.6499


ValueError: too many values to unpack (expected 3)

In [ ]:
ckpt = torch.load("checkpoints/CVAE/best.pt", map_location=device)
model.load_state_dict(ckpt["generator"])
model.eval()
print(f"Bestes Modell aus Epoch {ckpt['epoch'] + 1} geladen.")
 
# ---- Metriken über das ganze Test-Set ----
psnr_total, ssim_total, n = 0.0, 0.0, 0
with torch.no_grad():
    for cloudy, target, mask in test_dataloader:
        cloudy, target, mask = cloudy.to(device), target.to(device), mask.to(device)
        if use_masks:
            cloudy_with_mask = torch.cat([cloudy, mask], dim=1)
            output_gen = model(cloudy_with_mask)
        else:
            output_gen = model(cloudy)
        for i in range(output_gen.size(0)):
            out_np, gt_np = to_img(output_gen[i]), to_img(target[i])
            psnr_total += psnr_fn(gt_np, out_np, data_range=1.0)
            ssim_total += ssim_fn(gt_np, out_np, data_range=1.0, channel_axis=2)
            n += 1
 
test_psnr = psnr_total / n
test_ssim = ssim_total / n
print(f"TEST  PSNR: {test_psnr:.2f} dB | SSIM: {test_ssim:.4f}  (über {n} Bilder)")
 

 
plot_test_samples(
    model, test_dataloader, device,
    n_show=5, save_path="checkpoints/test_samples.png",
)